# Week 10: Transport Networks & Accessibility Analysis

**Connection to QGIS:** This notebook mirrors the accessibility analysis from **QGIS Week 6**.

| What we do in QGIS (Week 6) | What we do in Python (this notebook) |
|----------------------------|--------------------------------------|
| QuickOSM plugin to download roads | `osmnx.graph_from_point()` |
| QNEAT3 plugin for network analysis | `networkx` graph algorithms |
| QNEAT3 Iso-Area for isochrones | `nx.ego_graph()` + convex hull |
| Add vector layer (facilities) | `gpd.read_file()` |
| Export as GeoPackage | `gdf.to_file()` |

---

## What you'll learn

1. **Network data concepts** - nodes (intersections) and edges (road segments)
2. **OSMnx** - Download real street networks from OpenStreetMap
3. **NetworkX** - Analyze networks (shortest paths, accessibility)
4. **Isochrones** - Map areas reachable within X minutes
5. **Export for QGIS** - Use Python results in your desktop GIS

---

## Why Python for network analysis?

| QGIS approach | Python approach |
|---------------|------------------|
| Manual download, install plugins | Single script, reproducible |
| Click-based analysis | Code-based, automatable |
| One location at a time | Batch process many locations |
| Results stay in project | Results exportable anywhere |

**Best practice:** Use Python for bulk analysis, QGIS for final map design.

---

## Step 0: Environment setup

This cell detects whether you're running in Google Colab or locally.

**QGIS equivalent:** Installing QGIS plugins (QuickOSM, QNEAT3)

In [ ]:
# =============================================================================
# ENVIRONMENT DETECTION
# =============================================================================
# This code checks if we're in Colab (cloud) or local (your computer)
# It's the same pattern used in all our notebooks

import sys

# Check if Google Colab's module exists in the system
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # ---------------------------------------------------------------------
    # COLAB: Install packages that aren't pre-installed
    # ---------------------------------------------------------------------
    # Colab has basic packages but not GIS-specific ones
    # The -q flag means "quiet" (less output)
    print("Running in Google Colab")
    print("Installing GIS packages (1-2 minutes)...")
    !pip install geopandas osmnx networkx -q
    print("Installation complete!")
else:
    # ---------------------------------------------------------------------
    # LOCAL: Assume conda environment is activated
    # ---------------------------------------------------------------------
    # If you get import errors, make sure you ran:
    # conda activate intro-gis
    print("Running locally")
    print("Make sure you activated: conda activate intro-gis")

---

## Step 1: Set up folder structure

**Same folder pattern as QGIS:**
- `raw/` = downloaded or input data (read-only)
- `processed/` = your analysis outputs

This notebook downloads network data from OpenStreetMap, so you don't need any input files to start. However, if you want to analyze specific facility locations (hospitals, schools, etc.), place them in `data/raw/`.

In [ ]:
# =============================================================================
# PATH CONFIGURATION
# =============================================================================
# Set up file paths that work in both Colab and local environments

from pathlib import Path

if IN_COLAB:
    # -------------------------------------------------------------------------
    # COLAB: Mount Google Drive to access your files
    # -------------------------------------------------------------------------
    # This pops up a permissions dialog - click "Connect to Google Drive"
    # Your Drive appears at /content/drive/MyDrive/
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Set paths to your Google Drive folders
    RAW = Path("/content/drive/MyDrive/intro-gis/week10/data/raw")
    PROCESSED = Path("/content/drive/MyDrive/intro-gis/week10/data/processed")
else:
    # -------------------------------------------------------------------------
    # LOCAL: Use folders relative to this notebook
    # -------------------------------------------------------------------------
    # ../data/ means "go up one folder, then into data/"
    RAW = Path("../data/raw")
    PROCESSED = Path("../data/processed")

# Create folders if they don't exist
# parents=True creates parent folders too
# exist_ok=True won't error if folder already exists
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

print(f"Raw data folder:       {RAW.resolve()}")
print(f"Processed data folder: {PROCESSED.resolve()}")

---

## Step 2: Import libraries

| Library | Purpose | QGIS equivalent |
|---------|---------|------------------|
| `geopandas` | Read/write spatial data | Add Vector Layer, Save As |
| `osmnx` | Download OpenStreetMap data | QuickOSM plugin |
| `networkx` | Network/graph analysis | QNEAT3 plugin |
| `matplotlib` | Visualize results | Print Layout |
| `shapely` | Create geometries | Sketching/digitizing |

In [ ]:
# =============================================================================
# IMPORT LIBRARIES
# =============================================================================
# Each library has a specific purpose for network analysis

import geopandas as gpd       # Spatial data handling (like QGIS layers)
import networkx as nx          # Network/graph algorithms (like QNEAT3)
import osmnx as ox             # Download OSM street networks (like QuickOSM)
import matplotlib.pyplot as plt # Plotting (like Print Layout)
from shapely.geometry import Point, Polygon  # Create geometries

# -----------------------------------------------------------------------------
# OSMNX SETTINGS
# -----------------------------------------------------------------------------
# Enable caching so repeated downloads use saved data
# This makes re-running cells much faster
ox.settings.use_cache = True
ox.settings.log_console = False  # Reduce verbose output

print("Libraries imported successfully!")
print(f"OSMnx version: {ox.__version__}")

---

## Step 3: Download street network from OpenStreetMap

**QGIS equivalent:** Using the QuickOSM plugin to download roads

OSMnx downloads real street data directly from OpenStreetMap servers:
- No manual download needed
- Data is always up-to-date
- Automatically converts to a network graph

### Key parameters

| Parameter | Description | Example |
|-----------|-------------|----------|
| `(lat, lon)` | Center point coordinates | `(-33.8688, 151.2093)` for Sydney |
| `dist` | Radius in meters | `1000` = 1km around center |
| `network_type` | What kind of routes | `"walk"`, `"drive"`, `"bike"` |

**Tip:** Right-click in Google Maps to copy coordinates for any location.

In [ ]:
# =============================================================================
# DEFINE STUDY AREA
# =============================================================================
# Change these coordinates to analyze any location!
# Right-click in Google Maps to copy coordinates

# Example: University of Sydney (change to your study area)
LATITUDE = -33.8886      # Negative for Southern Hemisphere
LONGITUDE = 151.1873     # Positive for Eastern Hemisphere
RADIUS = 1000            # Meters around center point (1km)

# Network type determines which roads are included:
#   "walk"  = footpaths, pedestrian areas, roads (excludes motorways)
#   "drive" = roads suitable for cars (excludes footpaths)
#   "bike"  = roads suitable for cycling
#   "all"   = all road types
NETWORK_TYPE = "walk"

print(f"Study area: {RADIUS}m radius around ({LATITUDE}, {LONGITUDE})")
print(f"Network type: {NETWORK_TYPE}")

In [ ]:
# =============================================================================
# DOWNLOAD STREET NETWORK
# =============================================================================
# This downloads real street data from OpenStreetMap
# Takes 5-30 seconds depending on area size and internet speed
#
# QGIS equivalent: QuickOSM plugin > Quick Query > highway

print(f"Downloading {NETWORK_TYPE} network...")
print("(This takes 5-30 seconds depending on area size)")

# graph_from_point() downloads and builds a network graph
# A "graph" is the data structure for networks:
#   - NODES = intersection points (where streets meet)
#   - EDGES = street segments (the roads between intersections)
G = ox.graph_from_point(
    center_point=(LATITUDE, LONGITUDE),  # Where to center the download
    dist=RADIUS,                          # How far around center (meters)
    network_type=NETWORK_TYPE             # What type of routes
)

# Print network statistics
print(f"\nDownload complete!")
print(f"Nodes (intersections):    {len(G.nodes):,}")
print(f"Edges (street segments):  {len(G.edges):,}")
print(f"\nThis is like having a road shapefile + topology built in!")

---

## Step 4: Visualize the network

**QGIS equivalent:** Styling a road layer to see the network

The network has two components:
- **Nodes** (dots) = intersections where streets meet
- **Edges** (lines) = street segments connecting intersections

Think of it like a subway map: stations are nodes, tracks are edges.

In [ ]:
# =============================================================================
# VISUALIZE THE NETWORK
# =============================================================================
# ox.plot_graph() creates a quick visualization of the network
#
# QGIS equivalent: Adding road layer and styling it

fig, ax = ox.plot_graph(
    G,                      # The network graph we downloaded
    figsize=(10, 10),       # Figure size in inches
    node_size=5,            # Size of intersection dots (0 hides them)
    node_color="red",       # Color of nodes
    edge_linewidth=0.5,     # Thickness of street lines
    edge_color="gray",      # Color of streets
    bgcolor="white"         # Background color
)

print("Street network visualization")
print("Red dots = intersections (nodes)")
print("Gray lines = streets (edges)")

---

## Step 5: Define facility location(s)

**QGIS equivalent:** Adding a point layer with facility locations

A "facility" is any location you want to measure accessibility FROM:
- Hospital / clinic
- School / university
- Train station / bus stop
- Park / community center

### Two options:

1. **Use network center** (default) - good for exploring
2. **Load your own file** - place `facilities.geojson` in `data/raw/`

**To create facilities.geojson in QGIS:**
1. Create new shapefile layer (points)
2. Add points at your facility locations
3. Export as GeoJSON with CRS EPSG:4326

In [ ]:
# =============================================================================
# LOAD OR CREATE FACILITY LOCATIONS
# =============================================================================
# Check if user has their own facilities file, otherwise use sample data
#
# QGIS equivalent: Layer > Add Layer > Add Vector Layer

# Path to optional user-provided facilities file
facilities_path = RAW / "facilities.geojson"

if facilities_path.exists():
    # -------------------------------------------------------------------------
    # OPTION 1: Load user's facilities from file
    # -------------------------------------------------------------------------
    print(f"Found facilities file: {facilities_path}")
    facilities = gpd.read_file(facilities_path)
    
    # Ensure CRS is WGS84 (EPSG:4326) to match our network
    # to_crs() reprojects if needed (like QGIS "Reproject Layer")
    facilities = facilities.to_crs("EPSG:4326")
    
    print(f"Loaded {len(facilities)} facility location(s)")
    
else:
    # -------------------------------------------------------------------------
    # OPTION 2: Create sample facility at network center
    # -------------------------------------------------------------------------
    print("No facilities.geojson found in data/raw/")
    print("Creating sample facility at network center...")
    
    # Create a GeoDataFrame with one point
    # This is like creating a new point layer in QGIS
    facilities = gpd.GeoDataFrame(
        # Attribute data (like the attribute table)
        {"name": ["Study Area Center"], 
         "type": ["Sample"]},
        # Geometry column with a Point object
        geometry=[Point(LONGITUDE, LATITUDE)],  # Note: Point(x, y) = Point(lon, lat)
        # Coordinate Reference System
        crs="EPSG:4326"  # WGS84 (same as Google Maps)
    )
    
    print(f"\nTo use your own facilities:")
    print(f"1. Create a point GeoJSON in QGIS")
    print(f"2. Save to: {RAW}/facilities.geojson")
    print(f"3. Re-run this cell")

# Display the facilities table
print(f"\nFacilities:")
facilities

---

## Step 6: Calculate walking time isochrones

**QGIS equivalent:** QNEAT3 plugin > Iso-Area as Polygons

### What is an isochrone?

An **isochrone** (Greek: iso=equal, chronos=time) shows all locations reachable within a certain travel time.

| Isochrone | Walking distance | Real-world meaning |
|-----------|------------------|--------------------|
| 5 min | ~400m | Very convenient access |
| 10 min | ~800m | Reasonable walking distance |
| 15 min | ~1200m | Maximum comfortable walk |
| 20 min | ~1600m | At edge of walkability |

### How we calculate isochrones:

1. **Find nearest node** - Snap facility to nearest intersection
2. **Calculate distance** - time × walking speed = distance in meters
3. **Find reachable nodes** - All intersections within that network distance
4. **Draw boundary** - Create a polygon around reachable area

In [ ]:
# =============================================================================
# ISOCHRONE PARAMETERS
# =============================================================================
# Configure walking speed and time intervals

# Walking speed assumptions
# Average adult walking speed is 4.5-5.0 km/h
# We use 4.8 km/h (80 meters per minute) as a reasonable default
WALK_SPEED_KMH = 4.8
METERS_PER_MINUTE = WALK_SPEED_KMH * 1000 / 60  # Convert to m/min

print(f"Walking speed: {WALK_SPEED_KMH} km/h = {METERS_PER_MINUTE:.0f} meters/minute")

# Time thresholds for isochrones (in minutes)
# Each value creates one isochrone ring
TIME_BREAKS = [5, 10, 15]  # 5-minute, 10-minute, 15-minute zones

print(f"\nIsochrone time breaks: {TIME_BREAKS} minutes")
print(f"This will create {len(TIME_BREAKS)} zones per facility")

In [ ]:
# =============================================================================
# CALCULATE ISOCHRONES
# =============================================================================
# For each facility and time break, calculate the reachable area
#
# QGIS equivalent: QNEAT3 > Iso-Area as Polygons (from Layer)

# List to store all isochrone polygons
isochrones = []

# Loop through each facility
for idx, facility in facilities.iterrows():
    facility_name = facility.get("name", f"Facility {idx}")
    print(f"\nProcessing: {facility_name}")
    
    # -------------------------------------------------------------------------
    # STEP 1: Find the nearest network node to this facility
    # -------------------------------------------------------------------------
    # The facility point probably isn't exactly on a road
    # We "snap" it to the nearest intersection (node)
    # This is like "Snap to Layer" in QGIS
    
    nearest_node = ox.distance.nearest_nodes(
        G,                    # The network graph
        facility.geometry.x,  # Longitude of facility
        facility.geometry.y   # Latitude of facility
    )
    
    # Loop through each time threshold
    for minutes in TIME_BREAKS:
        # ---------------------------------------------------------------------
        # STEP 2: Calculate maximum walking distance for this time
        # ---------------------------------------------------------------------
        max_distance = minutes * METERS_PER_MINUTE
        print(f"  {minutes} min = {max_distance:.0f}m network distance")
        
        # ---------------------------------------------------------------------
        # STEP 3: Find all nodes reachable within this distance
        # ---------------------------------------------------------------------
        # nx.ego_graph() finds all nodes within a certain "radius" of a starting node
        # The "radius" here is network distance (along roads), not straight-line
        # distance="length" means use the "length" attribute of edges (in meters)
        
        # Get subgraph of all reachable nodes
        subgraph = nx.ego_graph(
            G,                    # Full network
            nearest_node,         # Starting node
            radius=max_distance,  # Maximum distance
            distance="length"     # Use edge length for distance
        )
        
        # ---------------------------------------------------------------------
        # STEP 4: Create a polygon around the reachable nodes
        # ---------------------------------------------------------------------
        # Convert the subgraph nodes to a GeoDataFrame (points)
        # Then create a convex hull (smallest polygon containing all points)
        
        # graph_to_gdfs() converts network to GeoDataFrames
        # edges=False means we only want nodes, not edges
        nodes_gdf = ox.graph_to_gdfs(subgraph, edges=False)
        
        # unary_union combines all points into a single geometry
        # convex_hull creates the boundary polygon
        # (Like QGIS: Vector > Geoprocessing > Convex Hull)
        reachable_area = nodes_gdf.unary_union.convex_hull
        
        # Store this isochrone with its attributes
        isochrones.append({
            "facility": facility_name,
            "minutes": minutes,
            "distance_m": max_distance,
            "nodes_reached": len(subgraph.nodes),
            "geometry": reachable_area
        })

# Convert list to GeoDataFrame
# CRS comes from the network graph (usually EPSG:4326)
iso_gdf = gpd.GeoDataFrame(isochrones, crs=G.graph["crs"])

print(f"\nCreated {len(iso_gdf)} isochrones total")
iso_gdf

---

## Step 7: Map the isochrones

**QGIS equivalent:** Styling layers with graduated colors

We'll create a layered map showing:
- Isochrone zones (colored by time)
- Street network (for context)
- Facility location (red dot)

In [ ]:
# =============================================================================
# CREATE ISOCHRONE MAP
# =============================================================================
# Plot isochrones with the network and facility
#
# QGIS equivalent: Styling multiple layers with categorized symbology

# Create figure and axis
fig, ax = plt.subplots(figsize=(12, 12))

# -------------------------------------------------------------------------
# Define colors for each time threshold
# -------------------------------------------------------------------------
# Green = close/accessible, Yellow = moderate, Orange/Red = further
# This is a common accessibility color scheme
colors = {
    5: "#2ecc71",   # Green - 5 minute walk
    10: "#f1c40f",  # Yellow - 10 minute walk  
    15: "#e67e22"   # Orange - 15 minute walk
}

# -------------------------------------------------------------------------
# Plot isochrones (largest first so smaller ones appear on top)
# -------------------------------------------------------------------------
# reversed() so 15-min plots first (underneath), then 10, then 5 on top

for minutes in reversed(TIME_BREAKS):
    # Filter to just this time threshold
    subset = iso_gdf[iso_gdf["minutes"] == minutes]
    
    # Plot with semi-transparency (alpha)
    subset.plot(
        ax=ax,
        color=colors.get(minutes, "gray"),  # Get color from dict
        alpha=0.5,                           # 50% transparent
        edgecolor="black",                   # Black outline
        linewidth=1,
        label=f"{minutes} min walk"          # For legend
    )

# -------------------------------------------------------------------------
# Plot street network for context
# -------------------------------------------------------------------------
# Convert network edges to GeoDataFrame for plotting
_, edges_gdf = ox.graph_to_gdfs(G)
edges_gdf.plot(ax=ax, color="gray", linewidth=0.3, alpha=0.5)

# -------------------------------------------------------------------------
# Plot facility location(s)
# -------------------------------------------------------------------------
# zorder=5 puts this layer on top of everything else
facilities.plot(
    ax=ax,
    color="red",
    markersize=100,
    marker="*",      # Star symbol
    zorder=5,
    label="Facility"
)

# -------------------------------------------------------------------------
# Add map elements
# -------------------------------------------------------------------------
ax.legend(loc="upper right", fontsize=10)
ax.set_title("Walking Time Isochrones\n(Network-based accessibility)", fontsize=14)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

# Remove axis for cleaner look
ax.set_axis_off()

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Green zone: Within 5-minute walk (very accessible)")
print("- Yellow zone: 5-10 minute walk (accessible)")
print("- Orange zone: 10-15 minute walk (moderate accessibility)")

---

## Step 8: Calculate a shortest path

**QGIS equivalent:** QNEAT3 > Shortest Path

Beyond isochrones, we can calculate the actual shortest route between two points. This is useful for:
- Finding the best route to a facility
- Measuring exact travel distance/time
- Routing applications

In [ ]:
# =============================================================================
# CALCULATE SHORTEST PATH
# =============================================================================
# Find the shortest route between two points on the network
#
# QGIS equivalent: QNEAT3 > Shortest Path (Point to Point)

# Define origin and destination
# (Using the facility as origin, and a random point as destination)
origin_lat = LATITUDE
origin_lon = LONGITUDE

# Destination: offset from origin (within our network)
dest_lat = LATITUDE + 0.005   # ~500m north
dest_lon = LONGITUDE + 0.005  # ~500m east

# Find nearest network nodes to origin and destination
origin_node = ox.distance.nearest_nodes(G, origin_lon, origin_lat)
dest_node = ox.distance.nearest_nodes(G, dest_lon, dest_lat)

print(f"Origin node: {origin_node}")
print(f"Destination node: {dest_node}")

# Calculate shortest path using Dijkstra's algorithm
# weight="length" means we minimize distance (could also use "travel_time")
try:
    route = nx.shortest_path(
        G,
        origin_node,
        dest_node,
        weight="length"  # Minimize by distance
    )
    
    # Calculate total route length
    route_length = nx.shortest_path_length(G, origin_node, dest_node, weight="length")
    walking_time = route_length / METERS_PER_MINUTE
    
    print(f"\nRoute found!")
    print(f"Nodes in route: {len(route)}")
    print(f"Total distance: {route_length:.0f} meters")
    print(f"Walking time: {walking_time:.1f} minutes")
    
except nx.NetworkXNoPath:
    print("No path found! Origin and destination may not be connected.")
    route = None

In [ ]:
# =============================================================================
# VISUALIZE SHORTEST PATH
# =============================================================================
# Plot the route on the network

if route:
    fig, ax = ox.plot_graph_route(
        G, 
        route,
        route_color="red",
        route_linewidth=3,
        node_size=0,
        figsize=(10, 10),
        bgcolor="white"
    )
    print("Red line shows the shortest walking route")
else:
    print("Cannot visualize - no route was found")

---

## Step 9: Export results for QGIS

**QGIS equivalent:** Right-click layer > Export > Save Features As

Save our Python analysis results so we can:
- Open them in QGIS for professional styling
- Overlay with other datasets (population, land use)
- Include in map layouts

In [ ]:
# =============================================================================
# EXPORT ISOCHRONES
# =============================================================================
# Save as GeoPackage (preferred format for QGIS)
#
# GeoPackage (.gpkg) advantages:
# - Single file (unlike shapefile's 4+ files)
# - Supports long field names
# - Faster for large datasets
# - Open standard

output_path = PROCESSED / "isochrones.gpkg"
iso_gdf.to_file(output_path, driver="GPKG")

print(f"Saved isochrones to: {output_path}")
print(f"\nColumns in output:")
for col in iso_gdf.columns:
    print(f"  - {col}")

In [ ]:
# =============================================================================
# EXPORT NETWORK AS GEODATAFRAME (OPTIONAL)
# =============================================================================
# Save the street network for use in QGIS
# This gives you two layers: nodes (points) and edges (lines)

# Convert graph to GeoDataFrames
nodes_gdf, edges_gdf = ox.graph_to_gdfs(G)

# Save edges (streets) - more commonly needed
edges_output = PROCESSED / "street_network.gpkg"
edges_gdf.to_file(edges_output, driver="GPKG")
print(f"Saved street network to: {edges_output}")

# Save facilities
facilities_output = PROCESSED / "facilities_used.gpkg"
facilities.to_file(facilities_output, driver="GPKG")
print(f"Saved facilities to: {facilities_output}")

print(f"\nAll outputs saved to: {PROCESSED}")
print(f"\nTo open in QGIS:")
print(f"1. Layer > Add Layer > Add Vector Layer")
print(f"2. Navigate to {PROCESSED}")
print(f"3. Select .gpkg files")

---

## Summary: Python vs QGIS comparison

| Task | QGIS approach | Python approach |
|------|---------------|------------------|
| Download street network | QuickOSM plugin | `ox.graph_from_point()` |
| Calculate isochrones | QNEAT3 > Iso-Area | `nx.ego_graph()` + convex hull |
| Find shortest path | QNEAT3 > Shortest Path | `nx.shortest_path()` |
| Visualize network | Add layer + style | `ox.plot_graph()` |
| Export results | Save As | `gdf.to_file()` |

### When to use each:

| Use QGIS when... | Use Python when... |
|------------------|--------------------|
| Creating a single polished map | Analyzing many locations |
| Interactive exploration | Reproducible analysis |
| Manual editing needed | Automating workflows |
| Learning GIS concepts | Integrating with data science |

---

## Try it yourself

1. **Change the study area**: Update `LATITUDE`, `LONGITUDE` to your location
2. **Change network type**: Try `"drive"` or `"bike"` instead of `"walk"`
3. **Change time breaks**: Add `20, 25, 30` minute isochrones
4. **Load your own facilities**: Create `facilities.geojson` in QGIS
5. **Export and style in QGIS**: Open the GeoPackage outputs for professional maps

---

## Save your work

- **Colab:** File > Save a copy in Drive
- **Local:** Ctrl+S (Windows) or Cmd+S (Mac)